# inspecting datasets

In [2]:
from src.extract import extract, extract_all_datasets
import pandas as pd
import pathlib as pl

In [3]:
dataset_directory = pl.Path('datasets/brazilian-ecommerce/')
dataframes = extract_all_datasets(dataset_directory)

In [ ]:
for key, value in dataframes.items():
        # Investigating missing values

        # Getting datasets that has the null values
        # getting total column values for calculating total rows and columns
        column_with_nulls = value.columns[value.isna().any()].tolist()
        if column_with_nulls:
            print(key) # dataset name
            # Getting columns that has null valuesa
            for column_name in column_with_nulls:
                # Calculating percentage for null values
                null_counts = value[column_name].isna().sum()
                total_count = value.shape[0]
                null_value_percentage = (null_counts / total_count) * 100
                print(
                    f"{column_name}: {null_counts} missing values "
                    f"({null_value_percentage:.2f}%)"
                )

            print() # for new line

In [ ]:
 # Investigating orders dataset for order_delivered_customer_date, by checking orders status column and other things
orders_df = dataframes['olist_orders_dataset']

null_rows = orders_df[orders_df['order_delivered_customer_date'].isna()]

filtered_null_rows = null_rows.loc[
    :, 
        [
            
            'order_id', 
            'customer_id', 
            'order_status', 
            'order_approved_at',
            'order_purchase_timestamp', 
            'order_delivered_carrier_date', 'order_delivered_customer_date'
            
        ]
    ]

In [ ]:
# print(filtered_null_rows['order_status'].value_counts()) # check the order status wise counts

print(filtered_null_rows[filtered_null_rows['order_status'] == 'delivered'])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_delivered_carrier_date'].isna())])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_approved_at'].isna())])

# Task 06 - Investigate Product Information

In [ ]:
product_df = dataframes['olist_products_dataset']

In [ ]:
product_df

In [ ]:
filtered_products_df = product_df.loc[:, [
    'product_category_name', 
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]]

In [ ]:
display(filtered_products_df.head())

In [ ]:
missing_product_category_name = filtered_products_df[filtered_products_df['product_category_name'].isna()]

In [ ]:
missing_product_category_name.isna().sum().any()

In [ ]:
missing_products = filtered_products_df[filtered_products_df['product_category_name'].isna() & filtered_products_df['product_name_lenght'].isna() & filtered_products_df['product_description_lenght'].isna() & filtered_products_df['product_photos_qty'].isna()]

In [ ]:
missing_products

In [ ]:
product_df[product_df['product_id'].isna()]

In [ ]:
missing_dimensions = product_df[
    product_df[
        [
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ].isna().any(axis=1)
]

In [ ]:
missing_dimensions

In [ ]:
reviews_df = dataframes['olist_order_reviews_dataset']

In [ ]:
reviews_df.columns.tolist()

In [ ]:
reviews_df.shape

In [ ]:
# checking duplicates
reviews_df[reviews_df.duplicated()]

In [ ]:
# Checking missing records
reviews_df['review_score'].isna().sum()

In [ ]:
# checking unique 
reviews_df['review_score'].unique()

In [ ]:
# Scorewise occurances
reviews_df['review_score'].value_counts()

In [ ]:
reviews_df['review_id'].duplicated().sum()

In [ ]:
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Missing identifiers
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Duplicate identifiers
reviews_df[['review_id', 'order_id']].duplicated().sum()

In [ ]:
# Check review_id uniqueness
reviews_df['review_id'].nunique()

In [ ]:
# Review score distribution
reviews_df['review_score'].value_counts().sort_index()

In [ ]:
duplicate_reviews = reviews_df[
    reviews_df['review_id'].duplicated(keep=False)
]

duplicate_reviews.sort_values('review_id')

In [ ]:
duplicate_reviews['review_id'].value_counts().head(30)

In [ ]:
sample_review_id = duplicate_reviews['review_id'].value_counts().index[0]

duplicate_reviews[
    duplicate_reviews['review_id'] == sample_review_id
].T

In [ ]:
duplicate_reviews.groupby('review_id')['order_id'].nunique().value_counts()

In [ ]:
duplicate_reviews.groupby('review_id')['review_score'].nunique().value_counts()

# Transformation starts

- [x] Identify timestamp columns.
- [x] Understand their current Pandas data types.
- [x] Decide which columns need conversion.
- [x] Implement the conversion.
- [x] Verify the result.

In [ ]:
orders_df = dataframes['olist_orders_dataset']

In [ ]:
orders_df.dtypes

In [ ]:
orders_df.head()

In [ ]:
orders_df['order_status'].unique().tolist()

In [ ]:
orders_df['order_status'] = orders_df['order_status'].astype('category')

In [ ]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_delivered_carrier_date'] = pd.to_datetime(orders_df['order_delivered_carrier_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df[orders_df['order_purchase_timestamp'] == 'NaT']

# Task 08 - Investigate order timestamp consistency

### Part 1 — Investigate chronological consistency
- [x] Approval timestamp is earlier than purchase timestamp.
- [x] Carrier delivery timestamp is earlier than approval timestamp.
- [x] Customer delivery timestamp is earlier than carrier delivery timestamp.
- [x] Customer delivery timestamp is earlier than purchase timestamp.

In [ ]:
orders_df[orders_df['order_approved_at'] < orders_df['order_purchase_timestamp']].shape

In [ ]:
carrier_date_issued_data = orders_df[orders_df['order_delivered_carrier_date'] < orders_df['order_approved_at']]

In [ ]:
customer_date_issued_data = orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_delivered_carrier_date']]

In [ ]:
orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_purchase_timestamp']].shape

In [ ]:
orders_df.head()

### Part 2 — Investigate estimated delivery
- [x] Estimated delivery date is earlier than purchase date.
- [x] Actual customer delivery is later than the estimated delivery date.

In [ ]:
orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_purchase_timestamp']].shape

In [ ]:
estimated_delivery_date_issued_data = orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_delivered_customer_date']]

### Part 3 — Report your findings
- [x] Number of affected records.
- [x] A few sample records (order_id and relevant timestamps).

In [ ]:
orders_df[['order_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']]

In [ ]:
carrier_date_issued_data.head(10)

In [ ]:
customer_date_issued_data.head(10)

In [ ]:
estimated_delivery_date_issued_data.shape

In [ ]:
estimated_delivery_date_issued_data.head(10)

In [ ]:
carrier_date_issued_data['order_status'].value_counts()

In [ ]:
customer_date_issued_data['order_status'].value_counts()

In [ ]:
estimated_delivery_date_issued_data['order_status'].value_counts()

In [ ]:
total_delivered_orders_count = orders_df['order_status'].value_counts().tolist()[0]

In [ ]:
late_delivered_orders = estimated_delivery_date_issued_data['order_status'].value_counts().tolist()[0]

In [ ]:
late_delivery_percentage = (late_delivered_orders / total_delivered_orders_count) * 100

In [ ]:
round(late_delivery_percentage, 2)

## Import transform_order function

In [ ]:
from src.transform import transform_orders

In [ ]:
copied_order_df = transform_orders(orders_df)

In [ ]:
copied_order_df.dtypes

In [ ]:
orders_df.dtypes

In [ ]:
copied_order_df.shape

In [ ]:
orders_df.shape

In [ ]:
timestamp_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
    ]

In [ ]:
for column in timestamp_columns:
    print(
        column,
        orders_df[column].isna().sum(),
        copied_order_df[column].isna().sum()
    )

In [5]:
dataframes.keys()

dict_keys(['olist_sellers_dataset', 'product_category_name_translation', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset'])

In [ ]:
for key, value in dataframes.items():
    print(f'Dataset: {key}')
    for indx, column_name in enumerate(value.columns.tolist(), start=1):
        print(f"{indx} -> {column_name}")
    print()

# Task 13 A - Investigate order items

- [x] Data types of every column.
- [x] Missing values in every column.
- [x] Duplicate rows.
- [x] Whether order_id, order_item_id, and product_id contain missing values.
- [x] Whether price and freight_value contain zero or negative values.

In [11]:
order_items_df = dataframes['olist_order_items_dataset']

In [12]:
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [13]:
order_items_df.dtypes

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [14]:
order_items_df.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [16]:
duplicates = order_items_df.duplicated(keep=False)
order_items_df[duplicates]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [17]:
zero_or_negative = order_items_df['price'] <= 0
order_items_df[zero_or_negative]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [18]:
order_items_df[order_items_df['freight_value'] <= 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
...,...,...,...,...,...,...,...
111094,fc698f330ec7fb74859071cc6cb29772,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-04-25 02:31:57,53.9,0.0
111497,fd4907109f6bac23f07064af84bec02d,1,7a10781637204d8d10485c71a6108a2e,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-30 11:31:32,219.0,0.0
111649,fd95e4b85ebbb81853d4a6be3d61432b,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-05-04 11:10:31,106.9,0.0
112182,fee19a0dc7358b6962a611cecf6a37b4,1,f1c7f353075ce59d8a6f3cf58f419c9c,37be5a7c751166fbc5f8ccba4119e043,2017-09-07 22:06:31,195.0,0.0


In [19]:
(order_items_df['freight_value'] == 0).sum()

np.int64(383)

In [20]:
(order_items_df['freight_value'] < 0).sum()

np.int64(0)

In [21]:
order_items_df[order_items_df['freight_value'] <= 0].head(20)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-04-27 09:31:35,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-03 19:11:40,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-09 23:10:46,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-26 13:31:30,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-11 18:15:13,69.9,0.0
